<a href="https://colab.research.google.com/github/Violetaaa/tfg-learning/blob/main/03-langGraph_with_persistence_and_streaming.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install -q langgraph langchain-community langchain-google-genai

from langgraph.graph import StateGraph, END
from typing import TypedDict, Annotated
import operator
from langchain_core.messages import AnyMessage, SystemMessage, HumanMessage, ToolMessage
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_community.tools.tavily_search import TavilySearchResults

In [3]:
import os
from google.colab import userdata

os.environ["TAVILY_API_KEY"] = userdata.get('TAVILY_API_KEY')

tool = TavilySearchResults(max_results=4)
print(type(tool))
print(tool.name)

<class 'langchain_community.tools.tavily_search.tool.TavilySearchResults'>
tavily_search_results_json


/tmp/ipykernel_2975/577945945.py:6: LangChainDeprecationWarning: The class `TavilySearchResults` was deprecated in LangChain 0.3.25 and will be removed in 1.0. An updated version of the class exists in the `langchain-tavily package and should be used instead. To use it run `pip install -U `langchain-tavily` and import as `from `langchain_tavily import TavilySearch``.
  tool = TavilySearchResults(max_results=4)


In [4]:
class AgentState(TypedDict):
    messages: Annotated[list[AnyMessage], operator.add]

In [5]:
from langgraph.checkpoint.memory import MemorySaver

memory = MemorySaver()

In [6]:
class Agent:
    def __init__(self, model, tools, checkpointer, system=""):
        self.system = system
        graph = StateGraph(AgentState)
        graph.add_node("llm", self.call_openai)
        graph.add_node("action", self.take_action)
        graph.add_conditional_edges("llm", self.exists_action, {True: "action", False: END})
        graph.add_edge("action", "llm")
        graph.set_entry_point("llm")
        self.graph = graph.compile(checkpointer=checkpointer)
        self.tools = {t.name: t for t in tools}
        self.model = model.bind_tools(tools)

    def call_openai(self, state: AgentState):
        messages = state['messages']
        if self.system:
            messages = [SystemMessage(content=self.system)] + messages
        message = self.model.invoke(messages)
        return {'messages': [message]}

    def exists_action(self, state: AgentState):
        result = state['messages'][-1]
        return len(result.tool_calls) > 0

    def take_action(self, state: AgentState):
        tool_calls = state['messages'][-1].tool_calls
        results = []
        for t in tool_calls:
            print(f"Calling: {t}")
            result = self.tools[t['name']].invoke(t['args'])
            results.append(ToolMessage(tool_call_id=t['id'], name=t['name'], content=str(result)))
        print("Back to the model!")
        return {'messages': results}

In [18]:
import os
from google.colab import userdata

os.environ["GOOGLE_API_KEY"] = userdata.get('GOOGLE_API_KEY')

MODEL = "gemini-3-flash-preview"

prompt = """You are a smart research assistant. Use the search engine to look up information. \
You are allowed to make multiple calls (either together or in sequence). \
Only look up information when you are sure of what you want. \
If you need to look up some information before asking a follow up question, you are allowed to do that!
"""

model = ChatGoogleGenerativeAI(model=MODEL)
abot = Agent(model, [tool], system=prompt, checkpointer=memory)

In [13]:
messages = [HumanMessage(content="Who is the CEO of Alsa?")]

In [14]:
thread = {"configurable": {"thread_id": "1"}}

In [19]:
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v['messages'])

[AIMessage(content=[], additional_kwargs={'function_call': {'name': 'tavily_search_results_json', 'arguments': '{"query": "CEO of Alsa transport company"}'}, '__gemini_function_call_thought_signatures__': {'call_206962': 'EusDCugDAWkUfRPBD3vpOvkWL6WCDjX2JGcSK6gLCF/GZy/XVKUsdBEKo5y/kxThaOFiEE7CHqjRQC5zQcRorEi6bxFg0Koq1Jgg8mdglOxhRx9EZGOapEYOZCJ+odu5sHlDTjl7j53IHdUCrGQn8UV/8SDl7PqaeuR7RbuLOGQG7+CX+4ueX73rodXYkXYvVnRP6PNU0NFy3oQK1dZ2Zba7gSDth7xA0l3I1ydjggTljjaSgIgX3qHHxBzuDLFky16vdHWxAHwuNqVjfYvVmdnER6x6MB/Igp8NRTPVxzeeNSq672mjDt1AAcz/IbseMQ4igTNdcmJDTudZ3sF++OhIj1WZwSa7fnSVb3UUKeewrUJA8vu9P/vLz4R6rg7axdnN2FjFGxd+yNUjD+ma8jH9DFuGKYWAYboS9BWUF0kKTBDfp6+gHCwNhmWE+FMJl3V3Ba3Vn3EeLnphbXkcnmD71vMkH0LykcGAGjL0z33dYzn9pkt9xjHDkN5Ju3A5wKxY9M/1dkrU2My2AEOMWsj3/28DiVgbz1mSq7QMN2gl5Pnp2842W3QAQWquTMhZN4C7cj9oo858/kNcR9fuDHJ9w2SHcmmWJXjlPsMTX997P4HgzunTz8bHXJ2x/I2n5CRQczaHLRQfezd/v3g='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3-flash-preview', 'safety_ratings': [], 'model_

In [20]:
messages = [HumanMessage(content="y el de Alimerka?")]
thread = {"configurable": {"thread_id": "1"}}
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v)

{'messages': [AIMessage(content=[], additional_kwargs={'function_call': {'name': 'tavily_search_results_json', 'arguments': '{"query": "CEO de Alimerka actual"}'}, '__gemini_function_call_thought_signatures__': {'call_408336': 'EvsBCvgBAWkUfRPbRJdbwESYUuhodBLJ+QOj4PgwqcJ0z5ZxiZCej3FkHMR1rlcK8uEdriDuA+a36A6bwdO7qZ1yZH0QA2L3S4CMF82RDzoN70VPlPOXcdNIuV8Bg7pSpH9zAa58XxsbhjiQsQUKDVqHSHbw0sTTNzQlU3r527xFoWvefSAETsuRbkZRm7Uw4Qgr7tFFhYNo+E8qetLEbXcmMPvamxmbk1LVKJo7UEsvkJ4Dy7SgzvYVmLPRTu+8jeRFFqkGzlpXixMwJDKDD7J2qtNcgpO4Rd0TQUi4VyvSvDPlxhDQJV4xrwrCCi6upBpf2JFJEyFdoa3wCjE='}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3-flash-preview', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--01a12052-0ad1-7ac3-bd91-8bbc255e180a-0', tool_calls=[{'name': 'tavily_search_results_json', 'args': {'query': 'CEO de Alimerka actual'}, 'id': 'call_408336', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 414, 'output_tokens': 63, 'total_token

In [21]:
messages = [HumanMessage(content="¿Cuál lleva más tiempo en el cargo?")]
thread = {"configurable": {"thread_id": "1"}}
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v)

{'messages': [AIMessage(content=[{'type': 'text', 'text': 'El que lleva más tiempo en el cargo es **Alejandro Fernández González**, el presidente de **Alimerka**.\n\nAquí tienes la comparativa:\n\n*   **Alejandro Fernández (Alimerka):** Lleva en el cargo desde finales de **2008** (aproximadamente **16 años**), cuando asumió la presidencia tras el fallecimiento de su padre, Luis Noé Fernández.\n*   **Francisco Iglesias (Alsa):** Es el Consejero Delegado (CEO) desde **2015** (aproximadamente **9 años**).\n\nPor lo tanto, el presidente de Alimerka le saca unos 7 años de antigüedad en el puesto al responsable de Alsa.', 'extras': {'signature': 'EtIECs8EAWkUfROYSB46gkVjvbSLYHGcHSPkJI5N03JMSL/BdtZ/IqgVU6ku+Q6sysW9FQtI7P5c1cy4bKyUM7q8EYq6ZgxN+1JPqpQyvTT55k6TnP4DYzspNeYuee4wMq7zcCr2fYrdqHpISo4jzx4gcYJuxGPIC3RlrjyEAqDy3a5P/vHw6albPkYvxqXSCqrunfSLadNJDAh73nTiFq4FUAzABPz1UlflS2GWexl4Rnbf+rCBxcpgLShq9+pAJZjc6+ZtYmyhSHO8frGIUcIbACsxpaT03HgGcBejL/nceMPfBICCzGR58mblNrU+7rjGf2g4t3XTHL92i+m9UzunoExszVW

In [22]:
# NEW THREAD ID, no access to history
messages = [HumanMessage(content="¿Cuál lleva más tiempo en el cargo?")]
thread = {"configurable": {"thread_id": "2"}}
for event in abot.graph.stream({"messages": messages}, thread):
    for v in event.values():
        print(v)

{'messages': [AIMessage(content=[{'type': 'text', 'text': 'Para poder responderte, necesito saber **a quiénes te refieres**. \n\n¿Estás comparando a dos presidentes, alcaldes, directores técnicos o líderes mundiales en particular? \n\nSi te refieres a los **líderes mundiales actuales con más tiempo en el poder**, aquí tienes algunos ejemplos:\n\n1.  **Paul Biya (Camerún):** Es el presidente con más tiempo en el cargo (desde 1982, más de 41 años).\n2.  **Teodoro Obiang (Guinea Ecuatorial):** Es el líder con más tiempo total en el poder (desde 1979, 45 años), aunque empezó mediante un golpe de Estado.\n3.  **Denis Sassou Nguesso (República del Congo):** Suma más de 39 años en dos periodos distintos.\n\n**Si te refieres a candidatos de una elección actual o a dos personas específicas de tu país, por favor dime sus nombres y te daré el dato exacto.**', 'extras': {'signature': 'EtEOCs4OAWkUfRME3iMydYvBUfUvm/8tDDazxoMLz6sRYuGN+0gFYeH835lafaJYM8U3dJFf2qQvmzE72zFggNoUKnLYPbdt1E0j4Eim9H74RTw5nM

## Streaming tokens
Instead of waiting for the full answer (`invoke`), `astream_events` returns it chunk by chunk as the model generates it, like the typing effect in ChatGPT.
- `Calling:` / `Back to the model!` lines come from the Agent class (the llm → action → llm loop), not from streaming.
- Text separated by `|` is the final answer arriving in chunks.
- Barely noticeable in a notebook; useful in a chat UI for long answers

In [24]:
from langgraph.checkpoint.memory import MemorySaver

memory = MemorySaver()
abot = Agent(model, [tool], system=prompt, checkpointer=memory)

In [25]:
messages = [HumanMessage(content="Who is the CEO of Alsa?")]
thread = {"configurable": {"thread_id": "4"}}
# emit one event per internal step
async for event in abot.graph.astream_events({"messages": messages}, thread, version="v1"):
    kind = event["event"]
    # Keep only the text chunks generated by the model
    if kind == "on_chat_model_stream":
        content = event["data"]["chunk"].content
        if content: # empty = the model is requesting a tool, not writing text
          # Print each chunk as it arrives; "|" marks the boundary between chunks
            print(content, end="|")

/usr/local/lib/python3.13/dist-packages/IPython/core/interactiveshell.py:3551: LangChainDeprecationWarning: astream_events version='v1' is deprecated. Use version='v2' or astream instead.
  await eval(code_obj, self.user_global_ns, self.user_ns)


Calling: {'name': 'tavily_search_results_json', 'args': {'query': 'current CEO of Alsa bus company'}, 'id': 'call_249429', 'type': 'tool_call'}
Back to the model!
Calling: {'name': 'tavily_search_results_json', 'args': {'query': 'CEO of Alsa Spain Francisco Iglesias'}, 'id': 'call_311723', 'type': 'tool_call'}
Back to the model!
[{'type': 'text', 'text': 'The CEO of Alsa is **Francisco Iglesias**.', 'index': 0}]|[{'type': 'text', 'text': ' He has held this position for several years, leading the company as it operates as a major subsidiary of the Mobico Group (formerly known', 'index': 0}]|[{'type': 'text', 'text': ' as National Express Group).', 'index': 0}]|[{'type': 'text', 'text': '', 'extras': {'signature': 'EsUECsIEAWkUfRMRsCki1hbQN1UmtuLuzlUZWG3I2eEVz6wa7gmWlsBGJ/LNjQC6w6j1LEhxVVG7LA5W7J0AzfXm9AAqHC91TgiFxdUiC3HKSG0JXGA1IM3asZSIkF9MDo9KVbrmQSc4ugeXHrF9CZuKgD3RgCNpxgMvaf1YWPeAuaozdxS+/hQy4NUEbCg5FxTQIsiv5H27PuocM+yxwNj5CeCVs9sm4ChUjsSAUypKGay9ZjsJoI2FAv66yvqrYQUeKsUTdErYG5vapqNT+

In [11]:
from google import genai
client = genai.Client(api_key=os.environ["GOOGLE_API_KEY"])
for m in client.models.list():
    if "flash" in m.name:
        print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-flash-preview-tts
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-3.6-flash
models/gemini-3.7-flash
models/gemini-3.8-flash
models/gemini-3.1-flash-tts-preview
models/gemini-3.8-flash-tts
models/gemini-3.8-flash-lite-tts
models/gemini-2.5-flash-native-audio-latest
models/gemini-2.5-flash-native-audio-preview-09-2025
models/gemini-2.5-flash-native-audio-preview-12-2025
models/gemini-3.1-flash-live-preview
